# Control Matrix Asset Allocation Optimisation

Optimises a 2D control-matrix policy: allocation is a bilinearly interpolated function
of both time and wealth, parameterised on a grid of `TIME_NODE_COUNT x WEALTH_NODE_COUNT`.

This notebook uses:
- the same maximum time-node count as the time-based case,
- the same wealth-node construction as the wealth-based case,
- distributed initial wealth buckets (not single fixed initial wealth).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys
import math

np.random.seed(42)
torch.manual_seed(42)

sys.path.append('..')

from utils.experiment import (
    SimulationConfig,
    OptimizationResult,
    save_experiment,
    load_experiments,
)

from utils import (
    CholeskyBootstrapReturns,
    BlockBootstrapReturnsLoader,
    ControlMatrixPolicy,
    SigmoidWealthPenalty,
    simulate_wealth_trajectory,
    project_onto_simplex,
    project_policy_gradients_tangent_cone,
)
from utils.spending import (
    EXPENDITURE_GUIDELINES,
    DecliningRealSpending,
    DecliningRealFloor,
    NZSuper,
    SpendingPolicy,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")

GPU: NVIDIA GeForce RTX 3070
GPU Memory: 8.6 GB


## Configuration Parameters

In [ ]:
# ============================================================================
# SIMULATION PARAMETERS
# ============================================================================
N_ASSETS = 3
N_SIMULATIONS = 400_000
SIMULATION_YEARS = 35

# ============================================================================
# CONTROL MATRIX STRUCTURE
# ============================================================================
# Node-count sweep requested for control-matrix optimisation.
TIME_NODE_COUNTS = [2,3,6,10]
WEALTH_NODE_COUNTS = [2,3,6,10]

# Wealth-node construction from the wealth-based case.
MIN_WEALTH_NODE = 250_000
MAX_WEALTH_NODE = 1_750_000

# ============================================================================
# RETURN SAMPLER
# ============================================================================
RETURN_SAMPLERS = [
    "cholesky",
    "block_bootstrapped",
    "block_bootstrapped_1950",
]

# ============================================================================
# WEALTH & SPENDING PARAMETERS
# ============================================================================
# Distributed initial wealth buckets [500_000, 550_000, ..., 1_000_000].
INITIAL_WEALTH_MIN = 500_000
INITIAL_WEALTH_MAX = 1_000_000
WEALTH_STEP = 50_000

DESIRED_SPENDING = EXPENDITURE_GUIDELINES['choices_metro_couple']
SPENDING_DECLINE_RATE = 0.02
CONSUMPTION_FLOOR = EXPENDITURE_GUIDELINES['no_frills_metro_couple']
FLOOR_DECLINE_RATE = 0.00
INCOME_TYPE = "couple"  # "couple" | "single" | "single_sharing" | None

# ============================================================================
# OPTIMIZATION PARAMETERS
# ============================================================================
# Initial risky-asset weights [bonds_weight, stocks_weight] at each matrix node.
INITIAL_NODE_POLICY = [0.0, 0.0]

OPTIMIZER_TYPE = "ADAM"
INITIAL_LR = .0050
MIN_LR = 1e-4
MAX_ITERATIONS = 20_000
MOMENTUM = 0.9

# Batch size schedule: [(batch_size, learning_rate, lr_patience, min_iterations), ...]


BATCH_SIZE_SCHEDULE = [
    # size, learning_rate, lr_patience, min_iterations
    (50_000, .01, 150, 150),
    (50_000, .005, 150, 150),
    # (25_000, 2, 350, 350),
    (100_000, 0.0015, 150, 150),
    (100_000, 0.001, 100, 100),
    (400_000, 0.001, 40, None),
]

# Learning-rate schedule (decay within final phase)
LR_DECAY_FACTOR = 0.25
LR_PATIENCE = 40
LR_THRESHOLD = 1e-4

# Early stopping
STOPPING_PATIENCE = 60

# Progress reporting
PRINT_EVERY = 5
HISTORY_SAVE_FREQUENCY = 20

# ============================================================================
# OBJECTIVE FUNCTION PARAMETERS
# ============================================================================
WEALTH_PENALTY_STEEPNESS = 1e-3


## Load Data

In [3]:
tax_rates = np.loadtxt("../Data/IID Data/Final/tax_rates.csv", delimiter=",", skiprows=1)

return_sampler_loaders = {}
for sampler_name in RETURN_SAMPLERS:
    if sampler_name == "cholesky":
        exp_returns = np.loadtxt("../Data/IID Data/Final/expected_returns.csv", delimiter=",", skiprows=1)
        cov_matrix = np.loadtxt(
            "../Data/IID Data/Final/covariance.csv", delimiter=",", skiprows=1, usecols=range(1, N_ASSETS + 2)
        )
        return_sampler_loaders[sampler_name] = CholeskyBootstrapReturns(exp_returns, cov_matrix)
    elif sampler_name in {"block_bootstrapped", "block_bootstrapped_1950"}:
        return_sampler_loaders[sampler_name] = BlockBootstrapReturnsLoader(
            f"../Data/Returns/Final/{sampler_name}.npy"
        )
    else:
        raise ValueError(f"Unknown RETURN_SAMPLER: {sampler_name}")

## Optimisation

In [4]:
completed_runs = set()
try:
    existing_experiments = load_experiments(results_dir="Results")
    for payload in existing_experiments:
        cfg = payload["config"]
        completed_runs.add((cfg.RETURN_SAMPLER, int(cfg.TIME_NODE_COUNT), int(cfg.WEALTH_NODE_COUNT)))
    print(f"Loaded {len(completed_runs)} completed runs from Results/.")
except FileNotFoundError:
    print("Results/ not found yet - all runs will be executed.")
except Exception as exc:
    print(f"Could not read existing Results ({exc}); proceeding without skip cache.")

Results/ not found yet - all runs will be executed.


In [5]:
from IPython.display import clear_output
from wakepy import keep

wealth_penalty = SigmoidWealthPenalty(steepness=WEALTH_PENALTY_STEEPNESS)

# Validate batch schedule
if BATCH_SIZE_SCHEDULE:
    for stage_idx, (stage_batch_size, stage_lr, stage_lr_patience, stage_min_iterations) in enumerate(BATCH_SIZE_SCHEDULE):
        if stage_batch_size <= 0:
            raise ValueError(f"BATCH_SIZE_SCHEDULE[{stage_idx}] has non-positive batch size: {stage_batch_size}")
        if stage_batch_size > N_SIMULATIONS:
            raise ValueError(
                f"BATCH_SIZE_SCHEDULE[{stage_idx}] batch size ({stage_batch_size}) exceeds "
                f"N_SIMULATIONS ({N_SIMULATIONS})"
            )
        if stage_lr_patience <= 0:
            raise ValueError(
                f"BATCH_SIZE_SCHEDULE[{stage_idx}] has non-positive lr_patience: {stage_lr_patience}"
            )
        if stage_min_iterations is not None and stage_min_iterations <= 0:
            raise ValueError(
                f"BATCH_SIZE_SCHEDULE[{stage_idx}] has non-positive min_iterations: {stage_min_iterations}"
            )

# Distributed initial wealth tensor with near-equal counts per bucket.
wealth_levels = torch.arange(
    INITIAL_WEALTH_MIN, INITIAL_WEALTH_MAX + WEALTH_STEP, WEALTH_STEP, dtype=torch.float32
)
n_buckets = len(wealth_levels)
repeats = N_SIMULATIONS // n_buckets
remainder = N_SIMULATIONS % n_buckets
initial_wealth_tensor = torch.cat([
    wealth_levels.repeat(repeats),
    wealth_levels[:remainder],
]).to(DEVICE)

print(f"Initial wealth buckets: {wealth_levels.numpy()} ({n_buckets} levels, ~{repeats:,} sims each)")
print(f"Batch size schedule: {BATCH_SIZE_SCHEDULE}")

def _build_fixed_batch_indices(total_size: int, batch_size: int, device: torch.device):
    """Create deterministic contiguous batch index chunks for one stage."""
    if batch_size >= total_size:
        return None

    all_indices = torch.arange(total_size, device=device)
    return [
        all_indices[start:start + batch_size]
        for start in range(0, total_size, batch_size)
    ]

with keep.running():
    for RETURN_SAMPLER in RETURN_SAMPLERS:
        sampler = return_sampler_loaders[RETURN_SAMPLER]
        returns_sample, cumulative_inflation_sample = sampler.generate(N_SIMULATIONS, SIMULATION_YEARS)

        for WEALTH_NODE_COUNT in WEALTH_NODE_COUNTS:
            # Same wealth-node construction as wealth-based case.
            wealth_nodes = torch.logspace(
                math.log10(MIN_WEALTH_NODE), math.log10(MAX_WEALTH_NODE), WEALTH_NODE_COUNT
            )
            for TIME_NODE_COUNT in TIME_NODE_COUNTS:
                # logarithmic time nodes
                time_nodes = torch.logspace(math.log10(10), math.log10(SIMULATION_YEARS+10), steps=TIME_NODE_COUNT, base=10)-10
                time_nodes = torch.linspace(0, SIMULATION_YEARS, steps=TIME_NODE_COUNT)

                run_key = (RETURN_SAMPLER, int(TIME_NODE_COUNT), int(WEALTH_NODE_COUNT))
                if run_key in completed_runs:
                    print(
                        f"Skipping completed run: sampler={RETURN_SAMPLER}, "
                        f"TIME_NODE_COUNT={TIME_NODE_COUNT}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                    )
                    continue

                print(f"\n{'='*70}")
                print(
                    f"STARTING COMBINATION: sampler={RETURN_SAMPLER}, "
                    f"TIME_NODE_COUNT={TIME_NODE_COUNT}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                )
                print(f"Time nodes: {time_nodes.numpy()}")
                print(f"Wealth nodes: {wealth_nodes.numpy()}")
                print(f"{'='*70}")

                returns = torch.tensor(returns_sample - tax_rates, device=DEVICE)
                cumulative_inflation = torch.tensor(cumulative_inflation_sample, device=DEVICE)

                sim_config = SimulationConfig(
                    N_ASSETS=N_ASSETS,
                    N_SIMULATIONS=N_SIMULATIONS,
                    SIMULATION_YEARS=SIMULATION_YEARS,
                    RETURN_SAMPLER=RETURN_SAMPLER,
                    INITIAL_WEALTH=None,  # Distributed bucketed initial wealth
                    DESIRED_SPENDING=DESIRED_SPENDING,
                    SPENDING_DECLINE_RATE=SPENDING_DECLINE_RATE,
                    CONSUMPTION_FLOOR=CONSUMPTION_FLOOR,
                    FLOOR_DECLINE_RATE=FLOOR_DECLINE_RATE,
                    INCOME_TYPE=INCOME_TYPE,
                    INITIAL_POLICY=INITIAL_NODE_POLICY,
                    OPTIMIZER_TYPE=OPTIMIZER_TYPE,
                    TIME_NODE_COUNT=TIME_NODE_COUNT,
                    WEALTH_NODE_COUNT=WEALTH_NODE_COUNT,
                )

                desired_spending_pol = DecliningRealSpending(DESIRED_SPENDING, decline_rate=SPENDING_DECLINE_RATE)
                consumption_floor_pol = DecliningRealFloor(init_floor=CONSUMPTION_FLOOR, decline_rate=FLOOR_DECLINE_RATE)
                income = NZSuper(INCOME_TYPE)
                spending_policy = SpendingPolicy(
                    spending=desired_spending_pol,
                    floor=consumption_floor_pol,
                    income=income,
                )

                allocation_policy = ControlMatrixPolicy(
                    N_ASSETS,
                    N_SIMULATIONS,
                    time_nodes.clone(),
                    wealth_nodes.clone(),
                    DEVICE,
                )

                base_node = torch.tensor(INITIAL_NODE_POLICY, device=DEVICE, dtype=torch.float32)
                policy = (
                    base_node
                    .unsqueeze(0)
                    .unsqueeze(0)
                    .repeat(TIME_NODE_COUNT, WEALTH_NODE_COUNT, 1)
                    .requires_grad_(True)
                )

                # Set up optimizer and batch schedule
                if BATCH_SIZE_SCHEDULE:
                    batch_stage_idx = 0
                    current_batch_size, scheduled_lr, current_lr_patience, current_min_iterations = BATCH_SIZE_SCHEDULE[batch_stage_idx]
                    optimizer_lr = scheduled_lr
                else:
                    batch_stage_idx = -1
                    current_batch_size = N_SIMULATIONS
                    current_lr_patience = LR_PATIENCE
                    current_min_iterations = None
                    optimizer_lr = INITIAL_LR
                stage_start_iteration = 0
                stage_batch_chunks = _build_fixed_batch_indices(N_SIMULATIONS, current_batch_size, DEVICE)
                stage_batch_ptr = 0

                if OPTIMIZER_TYPE.upper() == "ADAM":
                    optimizer = torch.optim.Adam([policy], lr=optimizer_lr)
                else:
                    optimizer = torch.optim.SGD([policy], lr=optimizer_lr, momentum=MOMENTUM)
                    
                cost_history = []
                policy_history = []
                best_cost = float('inf')
                best_policy = policy.data.clone()
                iterations_without_improvement = 0
                iterations_without_lr_improvement = 0

                for i in range(MAX_ITERATIONS):
                    optimizer.zero_grad()

                    if current_batch_size < N_SIMULATIONS:
                        batch_idx = stage_batch_chunks[stage_batch_ptr]
                        stage_batch_ptr = (stage_batch_ptr + 1) % len(stage_batch_chunks)
                        batch_returns = returns[batch_idx]
                        batch_cumulative_inflation = cumulative_inflation[batch_idx]
                        batch_initial_wealth = initial_wealth_tensor[batch_idx]
                    else:
                        batch_returns = returns
                        batch_cumulative_inflation = cumulative_inflation
                        batch_initial_wealth = initial_wealth_tensor

                    wealth, consumption = simulate_wealth_trajectory(
                        returns=batch_returns,
                        cumulative_inflation=batch_cumulative_inflation,
                        allocation_policy=allocation_policy,
                        spending_policy=spending_policy,
                        initial_wealth=batch_initial_wealth,
                        policy_settings=policy,
                    )

                    cost = wealth_penalty.evaluate(wealth, consumption)
                    cost.backward()
                    project_policy_gradients_tangent_cone(policy)
                    optimizer.step()

                    with torch.no_grad():
                        policy.data = project_onto_simplex(policy.data).clamp(0, 1)
                    cost_item = cost.item()
                    cost_history.append(cost_item)

                    if (i + 1) % HISTORY_SAVE_FREQUENCY == 0:
                        policy_history.append(policy.data.cpu().numpy().copy())

                    if cost_item < best_cost:
                        best_cost = cost_item
                        best_policy = policy.detach().clone().cpu()
                        iterations_without_improvement = 0
                        iterations_without_lr_improvement = 0
                    else:
                        iterations_without_improvement += 1
                        iterations_without_lr_improvement += 1

                    if iterations_without_improvement >= STOPPING_PATIENCE and batch_stage_idx >= len(BATCH_SIZE_SCHEDULE) - 1:
                        print(f"\n{'='*70}")
                        print(f"EARLY STOPPING at iteration {i + 1}")
                        print(f"No improvement in {STOPPING_PATIENCE} iterations")
                        print(f"{'='*70}")
                        break

                    stage_iterations = (i + 1) - stage_start_iteration
                    min_iterations_reached = (
                        current_min_iterations is None or stage_iterations >= current_min_iterations
                    )
                    if iterations_without_lr_improvement >= current_lr_patience and min_iterations_reached:
                        used_batch_schedule_step = False

                        if BATCH_SIZE_SCHEDULE and batch_stage_idx < len(BATCH_SIZE_SCHEDULE) - 1:
                            batch_stage_idx += 1
                            current_batch_size, scheduled_lr, current_lr_patience, current_min_iterations = BATCH_SIZE_SCHEDULE[batch_stage_idx]
                            for param_group in optimizer.param_groups:
                                old_lr = param_group['lr']
                                param_group['lr'] = scheduled_lr

                            print(f"\n{'='*70}")
                            print(
                                f"Batch schedule advanced at iteration {i + 1}: "
                                f"stage {batch_stage_idx + 1}/{len(BATCH_SIZE_SCHEDULE)}, "
                                f"batch_size={current_batch_size:,}, lr={old_lr:.6f}->{scheduled_lr:.6f}, "
                                f"lr_patience={current_lr_patience}, min_iterations={current_min_iterations}"
                            )
                            print(f"{'='*70}")
                            used_batch_schedule_step = True
                            iterations_without_lr_improvement = 0
                            iterations_without_improvement = 0
                            best_cost = float('inf')
                            stage_start_iteration = i + 1
                            stage_batch_chunks = _build_fixed_batch_indices(N_SIMULATIONS, current_batch_size, DEVICE)
                            stage_batch_ptr = 0
                        else:
                            stop = False
                            for param_group in optimizer.param_groups:
                                old_lr = param_group['lr']
                                new_lr = old_lr * LR_DECAY_FACTOR
                                param_group['lr'] = new_lr
                                if new_lr < MIN_LR:
                                    print(f"\n{'='*70}")
                                    print(f"LR below minimum threshold at iteration {i + 1}: {new_lr:.6f} < {MIN_LR:.6f}")
                                    print(f"Stopping optimization.")
                                    print(f"{'='*70}")
                                    stop = True
                                    break
                            if stop:
                                break

                            print(f"LR reduced at iteration {i+1}: {old_lr:.6f} -> {new_lr:.6f}")
                            iterations_without_lr_improvement = 0

                        if used_batch_schedule_step:
                            continue

                    if (i + 1) % PRINT_EVERY == 0 or i == 0:
                        clear_output(wait=True)
                        current_lr = optimizer.param_groups[0]['lr']
                        cost_change = cost_history[-1] - cost_history[-2] if len(cost_history) > 1 else 0
                        if BATCH_SIZE_SCHEDULE:
                            stage_label = f"{batch_stage_idx + 1}/{len(BATCH_SIZE_SCHEDULE)}"
                        else:
                            stage_label = "full"

                        print(f"{'='*70}")
                        print(
                            f"OPTIMISING: sampler={RETURN_SAMPLER} | "
                            f"TIME_NODE_COUNT={TIME_NODE_COUNT} | WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                        )
                        print(
                            f"Iteration {i+1:,}/{MAX_ITERATIONS:,} ({(i+1)/MAX_ITERATIONS*100:.1f}%)  |  "
                            f"LR: {current_lr:.6f}  |  Batch: {current_batch_size:,} (stage {stage_label}, lr_patience={current_lr_patience}, min_iterations={current_min_iterations})"
                        )
                        print(f"{'='*70}")
                        print(f"Cost:                      {cost_item:.6f}")
                        print(f"Cost change:               {cost_change:.8f}")
                        print(f"Best Cost:                 {best_cost:.6f}")
                        print(f"Iterations w/o improve:    {iterations_without_improvement}/{STOPPING_PATIENCE}")
                        print(f"{'-'*70}")
                        batch_size_for_metrics = wealth.shape[0]
                        print(f"Mean consumption:          ${consumption.mean().item():,.0f}")
                        print(f"Mean terminal wealth:      ${wealth[:, -1].mean().item():,.0f}")
                        print(f"Bankruptcy rate:           {(wealth[:, -1] == 0).sum().item() / batch_size_for_metrics:.2%}")
                        bankruptcy_density = (wealth == 0).sum().item() / (batch_size_for_metrics * SIMULATION_YEARS)
                        floor_t = consumption_floor_pol.calculate_tensor(
                            wealth=wealth, cumulative_inflation=batch_cumulative_inflation
                        )
                        impoverishment_density = (consumption < floor_t).sum().item() / (batch_size_for_metrics * SIMULATION_YEARS)
                        print(f"Impoverishment density:    {impoverishment_density:.4%}")
                        print(f"Bankruptcy density:        {bankruptcy_density:.4%}")
                        print(f"{'-'*70}")
                        print("Current policy (% bonds/% stocks):")
                        mid_w_idx = WEALTH_NODE_COUNT // 2
                        for t_idx in range(TIME_NODE_COUNT):
                            age = int(time_nodes[t_idx].item()) + 65
                            low_bonds = policy[t_idx, 0, 0].item()
                            low_stocks = policy[t_idx, 0, 1].item()
                            mid_bonds = policy[t_idx, mid_w_idx, 0].item()
                            mid_stocks = policy[t_idx, mid_w_idx, 1].item()
                            high_bonds = policy[t_idx, -1, 0].item()
                            high_stocks = policy[t_idx, -1, 1].item()
                            print(f"  Age {age}: lowW {low_bonds:.2%}/{low_stocks:.2%}  midW {mid_bonds:.2%}/{mid_stocks:.2%}  highW {high_bonds:.2%}/{high_stocks:.2%}")
                        print(f"{'='*70}")

                with torch.no_grad():
                    policy.copy_(best_policy.to(policy.device))

                policy_history = np.array(policy_history)
                cost_history = np.array(cost_history)

                print(f"\n{'='*70}")
                print(
                    f"OPTIMISATION COMPLETE - sampler={RETURN_SAMPLER} "
                    f"| TIME_NODE_COUNT={TIME_NODE_COUNT} | WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                )
                print(f"Best cost: {best_cost:.6f}  |  Iterations: {len(cost_history):,}")
                print(f"{'='*70}")

                final_wealth, final_consumption = simulate_wealth_trajectory(
                    returns=returns,
                    cumulative_inflation=cumulative_inflation,
                    allocation_policy=allocation_policy,
                    spending_policy=spending_policy,
                    initial_wealth=initial_wealth_tensor,
                    policy_settings=policy,
                )

                result = OptimizationResult(
                    best_policy=best_policy.detach().cpu().numpy(),
                    best_utility=-best_cost,
                    policy_history=policy_history,
                    cost_history=cost_history,
                    wealth_simulated=final_wealth.detach().cpu().numpy(),
                    consumption_simulated=final_consumption.detach().cpu().numpy(),
                    cumulative_inflation=cumulative_inflation.detach().cpu().numpy(),
                    time_nodes=time_nodes.detach().cpu().numpy(),
                    wealth_nodes=wealth_nodes.detach().cpu().numpy(),
                )
                save_experiment(sim_config, result)
                completed_runs.add(run_key)


OPTIMISING: sampler=block_bootstrapped | TIME_NODE_COUNT=10 | WEALTH_NODE_COUNT=10
Iteration 310/20,000 (1.6%)  |  LR: 0.010000  |  Batch: 50,000 (stage 1/5, lr_patience=150, min_iterations=150)
Cost:                      -0.948186
Cost change:               -0.00068313
Best Cost:                 -0.948223
Iterations w/o improve:    70/60
----------------------------------------------------------------------
Mean consumption:          $337,176
Mean terminal wealth:      $6,244,047
Bankruptcy rate:           18.96%
Impoverishment density:    9.2748%
Bankruptcy density:        8.8514%
----------------------------------------------------------------------
Current policy (% bonds/% stocks):
  Age 65: lowW 0.00%/100.00%  midW 0.00%/69.82%  highW 0.00%/100.00%
  Age 68: lowW 0.00%/100.00%  midW 0.00%/82.99%  highW 0.00%/100.00%
  Age 72: lowW 0.00%/100.00%  midW 0.00%/86.35%  highW 0.00%/100.00%
  Age 76: lowW 0.00%/100.00%  midW 0.00%/97.35%  highW 0.00%/100.00%
  Age 80: lowW 0.00%/100.00%

c:\Users\CallumDavidson\Code\MADS\Code\Control Matrix Policy\..\utils\allocation.py:170: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
  elementwise box constraints ($0 \le x_i \le 1$) plus a row-wise simplex


KeyboardInterrupt: 

In [ ]:
torch.logspace(math.log10(1), math.log10(SIMULATION_YEARS+1), steps=6, base=10)-1

tensor([ 0.0000,  1.0477,  3.1930,  7.5858, 16.5809, 35.0000])

In [ ]:
torch.linspace(0, SIMULATION_YEARS, steps=8)

tensor([ 0.,  5., 10., 15., 20., 25., 30., 35.])